## OpenDartReader를 이용한 재무제표(감사보고서 데이터) 추출

`dart.finstate_all()` 함수를 사용하여 특정 회사의 전체 재무제표 데이터를 추출합니다.

In [4]:
import OpenDartReader
import pandas as pd

# 데이터프레임 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# OpenDartReader 객체 생성 (API Key 필요)
api_key = '888e4aca3aba61a62811ba87e0fc054e2d9f6ea0' # opendartreader.ipynb와 동일한 키 사용
dart = OpenDartReader(api_key)


In [5]:
def get_financial_statements(corp_code, bsns_year, reprt_code='11011', fs_div='CFS'):
    """
    단일 회사의 전체 재무제표(감사보고서 포함) 데이터를 추출하는 함수
    
    Args:
        corp_code (str): 회사 고유번호 또는 종목코드
        bsns_year (int/str): 사업연도 (예: 2024)
        reprt_code (str): 보고서 코드 (기본값: '11011' 사업보고서)
        fs_div (str): 재무제표 종류 (기본값: 'CFS' 연결재무제표, 'OFS' 개별재무제표)
    
    Returns:
        pd.DataFrame: 추출된 재무제표 데이터프레임
    """
    print(f"[{corp_code}] {bsns_year}년도 재무제표 추출 시도 중...")
    try:
        # OpenDartReader 내부에서 데이터가 없을 때 정수와 비교하는 버그를 피하기 위해 int로 변환
        fs_df = dart.finstate_all(corp_code, int(bsns_year), reprt_code=reprt_code, fs_div=fs_div)
        
        if fs_df is None or fs_df.empty:
            print("=> 데이터가 존재하지 않습니다 (비상장사이거나 해당 연도/보고서가 없을 수 있습니다).")
        else:
            print(f"=> 성공적으로 {len(fs_df)}건의 재무 데이터를 가져왔습니다.")
            
        return fs_df
    except Exception as e:
        print(f"=> 데이터를 가져오는 중 오류 발생: {e}")
        return None


### 실행 테스트: 구다이글로벌 (01753163)

In [7]:
corp_code = '01753163' # 구다이글로벌
bsns_year = '2024'     # 2024년 사업연도

# 연결 재무제표(CFS) 추출 시도
df_cfs = get_financial_statements(corp_code, bsns_year, fs_div='CFS')
if df_cfs is not None and not df_cfs.empty:
    display(df_cfs.head())

# 연결 데이터가 없다면 별도(개별) 재무제표(OFS) 추출 시도
if df_cfs is None or df_cfs.empty:
    print("\n연결재무제표가 없으므로 개별재무제표(OFS) 추출을 시도합니다.")
    df_ofs = get_financial_statements(corp_code, bsns_year, fs_div='OFS')
    if df_ofs is not None and not df_ofs.empty:
        display(df_ofs.head())


[01753163] 2024년도 재무제표 추출 시도 중...
reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

=> 데이터가 존재하지 않습니다 (비상장사이거나 해당 연도/보고서가 없을 수 있습니다).

연결재무제표가 없으므로 개별재무제표(OFS) 추출을 시도합니다.
[01753163] 2024년도 재무제표 추출 시도 중...
reprt_code='11011', fs_div='OFS' (사업보고서, 별도(개별)제무제표)'
{'status': '013', 'message': '조회된 데이타가 없습니다.'}

=> 데이터가 존재하지 않습니다 (비상장사이거나 해당 연도/보고서가 없을 수 있습니다).
